# Gesture prediction + Egyptian Arabic voice demo

Pipeline: **gestures** (`CombinedGesture_model.pkl` + `Modelscaler.pkl` + `label_encoder.pkl`) → **Egyptian voice text** → **XTTS** (`Egyptian_Arabic_XTTS_Model/`) and **Laghtna Chatterbox** (`laghtna-chatterbox-v1 model/`, req. `pip install chatterbox-tts`).

**VRAM:** Never keep XTTS + Chatterbox on GPU together. Load → synth → `del model`; `gc.collect()`; `torch.cuda.empty_cache()`.
Neural synth uses **`torch.inference_mode()`**.

**NumPy / deps:** `CombinedGesture_model.pkl` needs **NumPy 2.x** (PCG64). **Coqui XTTS** in this notebook expects **`transformers==4.40.x`** (TTS imports `BeamSearchScorer`, removed from `transformers` 5.x top-level). That pin can make **`chatterbox-tts` fail to import** (it wants `transformers` 5.x) — XTTS is prioritized here.

**CSV:** `utf-8-sig` when reading the dataset avoids a **BOM-prefixed `flex1`** column name.

**Errors:** If XTTS/Chatterbox fail, sections print the traceback and execution continues (`nbconvert`/`Run All` completes). Missing `pip install TTS` / `pip install chatterbox-tts` is surfaced explicitly.


## Environment bootstrap (`run-once`)

**Why:** gesture `joblib.load` aligns with **NumPy 2.x** PCG64 blobs, but `pip install TTS` can pin **NumPy 1.26** again.

1. Run the next cell **once**.
2. **Restart the kernel.**
3. **Run All** from the imports/setup cell onward (skip `%pip` on daily runs once versions are steady).

**Chatterbox/perth pitfalls:** uninstall conflicting PyPI package `pip uninstall -y perth` → use `pip install resemble-perth` + `pip install 'setuptools<81'` (`pkg_resources` bridge).

In [1]:
import os, subprocess, sys

if os.environ.get("DEMO_NB_SKIP_BOOTSTRAP") == "1":
    print("[bootstrap] skipped (DEMO_NB_SKIP_BOOTSTRAP=1)")
else:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "perth"], check=False)
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "setuptools<81",
            "resemble-perth",
            "TTS",
            "chatterbox-tts",
            "huggingface_hub",
            "safetensors",
            "soundfile",
            "scipy",
            "scikit-learn",
            "joblib",
            "ipython",
            "jupyter",
            "nbconvert",
            "notebook",
            "ipykernel",
            "torch",
            "torchaudio",
        ]
    )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "numpy>=2.1,<2.2"]
    )
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--force-reinstall",
            "scipy",
            "scikit-learn",
            "joblib",
        ]
    )
    # pandas 2.2.x: NumPy-2 wheels + satisfies TTS internal `import pandas`.
    # transformers 4.40.x: Coqui TTS 0.22 XTTS layer still imports BeamSearchScorer from transformers.
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--force-reinstall",
            "pandas==2.2.3",
            "transformers==4.40.2",
        ]
    )
    print(
        "\nIMPORTANT: If pip changed NumPy, RESTART KERNEL once, then Run All from imports.\n"
        "Automated runs: set DEMO_NB_SKIP_BOOTSTRAP=1 after deps are stable.\n"
        "Note: transformers 4.40 is for XTTS; Chatterbox may not import until you use a separate env with transformers 5.x."
    )


[bootstrap] skipped (DEMO_NB_SKIP_BOOTSTRAP=1)


In [2]:
import gc
import csv
import json
import time
import traceback
from pathlib import Path

import joblib
import numpy as np

BASE_DIR = Path.cwd().resolve()
print("=" * 72)
print("cwd:", BASE_DIR)

FEATURE_COLS = [
    "flex1", "flex2", "flex3", "flex4", "flex5",
    "accX", "accY", "accZ",
    "gyroX", "gyroY", "gyroZ",
]

REQUIRED = {
    "CombinedGesture_model.pkl": BASE_DIR / "CombinedGesture_model.pkl",
    "label_encoder.pkl": BASE_DIR / "label_encoder.pkl",
    "Modelscaler.pkl": BASE_DIR / "Modelscaler.pkl",
    "numbers_letters_dataset_sorted.csv": BASE_DIR / "numbers_letters_dataset_sorted.csv",
    "Egyptian Arabic XTTS / config.json": BASE_DIR / "Egyptian_Arabic_XTTS_Model" / "config.json",
    "Egyptian Arabic XTTS / model.pth": BASE_DIR / "Egyptian_Arabic_XTTS_Model" / "model.pth",
    "Chatterbox / conds.pt": BASE_DIR / "laghtna-chatterbox-v1 model" / "conds.pt",
    "Chatterbox / t3_mtl23ls_v2": BASE_DIR / "laghtna-chatterbox-v1 model" / "t3_mtl23ls_v2.safetensors",
}

print("\nFILE CHECK:")
for lab, path in REQUIRED.items():
    ok = path.is_file()
    tag = "OK" if ok else "MISS"
    print(f" [{tag}] {lab}")

ref_nb = BASE_DIR / "CombinedDatasetModel.ipynb"
print("\nREFERENCE (not executed): CombinedDatasetModel.ipynb exists:", ref_nb.is_file())

try:
    import torch

    CUDA_AVAILABLE = torch.cuda.is_available()
    print("\nDEVICE")
    print(" CUDA available:", CUDA_AVAILABLE)
    if CUDA_AVAILABLE:
        print(" GPU name:", torch.cuda.get_device_name(0))
    DEVICE_PREF = "cuda" if CUDA_AVAILABLE else "cpu"
except ImportError:
    torch = None
    CUDA_AVAILABLE = False
    DEVICE_PREF = "cpu"
    print("\nWARN: torch not installed")

AUDIO_ROOT = BASE_DIR / "generated_audio"
XTTS_AUDIO_DIR = AUDIO_ROOT / "xtts"
CHATTER_AUDIO_DIR = AUDIO_ROOT / "chatterbox"
CACHE_DIR = AUDIO_ROOT / "_cache"
for d in (AUDIO_ROOT, XTTS_AUDIO_DIR, CHATTER_AUDIO_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_JSON = AUDIO_ROOT / "last_prediction_pipeline_state.json"
CHATTERBOX_DIR = BASE_DIR / "laghtna-chatterbox-v1 model"
XTTS_DIR = BASE_DIR / "Egyptian_Arabic_XTTS_Model"


def append_progress(payload: dict) -> None:
    prev = {}
    if INTERMEDIATE_JSON.exists():
        try:
            with open(INTERMEDIATE_JSON, encoding="utf-8") as fh:
                prev = json.load(fh)
        except Exception:
            prev = {}
    prev.update(payload)
    with open(INTERMEDIATE_JSON, "w", encoding="utf-8") as fh:
        json.dump(prev, fh, ensure_ascii=False, indent=2)


print("\ncheckpoint path:", INTERMEDIATE_JSON.resolve())
print("NumPy version (gesture pickles need 2.x):", np.__version__)


cwd: C:\Users\sondo\Desktop\Voice Model Stuff

FILE CHECK:
 [OK] CombinedGesture_model.pkl
 [OK] label_encoder.pkl
 [OK] Modelscaler.pkl
 [OK] numbers_letters_dataset_sorted.csv
 [OK] Egyptian Arabic XTTS / config.json
 [OK] Egyptian Arabic XTTS / model.pth
 [OK] Chatterbox / conds.pt
 [OK] Chatterbox / t3_mtl23ls_v2

REFERENCE (not executed): CombinedDatasetModel.ipynb exists: True



DEVICE
 CUDA available: False

checkpoint path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\last_prediction_pipeline_state.json
NumPy version (gesture pickles need 2.x): 2.4.5


In [3]:
def probe(mod: str, pip: str) -> bool:
    try:
        __import__(mod)
        print("[deps OK]", mod)
        return True
    except ImportError as e:
        print("[deps MISSING]", mod, "-> pip install", pip, "|", e)
        return False


probe("sklearn", "scikit-learn")
probe("pandas", "pandas")
HAVE_TORCH = probe("torch", "torch")
probe("torchaudio", "torchaudio")

HAVE_IP = probe("IPython.display", "ipython")
HAVE_TTS = probe("TTS", "TTS")

HAVE_SF = probe("soundfile", "soundfile")
HAVE_SAFE = probe("safetensors", "safetensors")

HAVE_CHATTERBOX = False
try:
    import chatterbox  # noqa: F401
    HAVE_CHATTERBOX = True
    print("[deps OK] chatterbox")
except ImportError:
    print("[deps MISSING] chatterbox -> pip install chatterbox-tts")

HAVE_HF = probe("huggingface_hub", "huggingface_hub")


[deps OK] sklearn
[deps OK] pandas
[deps OK] torch


[deps OK] torchaudio
[deps OK] IPython.display
[deps OK] TTS
[deps OK] soundfile
[deps OK] safetensors


[deps MISSING] chatterbox -> pip install chatterbox-tts
[deps OK] huggingface_hub


## Load gesture artefacts


In [4]:
clf = scaler = label_encoder = None
t0 = time.perf_counter()
try:
    clf = joblib.load(BASE_DIR / "CombinedGesture_model.pkl")
    scaler = joblib.load(BASE_DIR / "Modelscaler.pkl")
    label_encoder = joblib.load(BASE_DIR / "label_encoder.pkl")
except Exception:
    traceback.print_exc()
    print("\nBitGenerator pickle errors usually mean mismatched NumPy/Python — try Python 3.11 + matching sklearn trio.")
    raise

GESTURE_LOAD_S = time.perf_counter() - t0
print("Classifier:", type(clf).__name__)
print("#classes:", len(label_encoder.classes_))
print(f"Gesture bundle load timing: {GESTURE_LOAD_S:.4f}s")
append_progress({"gesture_load_s": GESTURE_LOAD_S})


Classifier: HistGradientBoostingClassifier
#classes: 39
Gesture bundle load timing: 0.8568s


## Dataset + sample row


In [5]:
# Dataset rows in memory (stdlib csv + NumPy only — stable with NumPy 2.x + TTS)
CSV_ROWS = []
_csv = BASE_DIR / "numbers_letters_dataset_sorted.csv"
if _csv.is_file():
    with open(_csv, newline="", encoding="utf-8-sig") as fh:
        rdr = csv.DictReader(fh)
        for row in rdr:
            rec = {k: (int(row[k]) if k != "label" else str(row[k]).strip()) for k in row}
            CSV_ROWS.append(rec)
    print("dataset rows:", len(CSV_ROWS), "| columns:", list(CSV_ROWS[0].keys()) if CSV_ROWS else [])
else:
    print("CSV missing")

FALLBACK = [
    {
        "flex1": 861,
        "flex2": 476,
        "flex3": 630,
        "flex4": 592,
        "flex5": 671,
        "accX": -8392,
        "accY": 14164,
        "accZ": 404,
        "gyroX": -606,
        "gyroY": -1314,
        "gyroZ": 272,
        "label": "أ",
    }
]


def row_to_matrix(rec: dict) -> np.ndarray:
    return np.array([[float(rec[c]) for c in FEATURE_COLS]], dtype=np.float64)


if CSV_ROWS:
    first_feat = row_to_matrix(CSV_ROWS[0])
else:
    first_feat = row_to_matrix(FALLBACK[0])

print("\nSample input (first row features), shape:", first_feat.shape)
print(first_feat)


dataset rows: 7800 | columns: ['flex1', 'flex2', 'flex3', 'flex4', 'flex5', 'accX', 'accY', 'accZ', 'gyroX', 'gyroY', 'gyroZ', 'label']

Sample input (first row features), shape: (1, 11)
[[  861.   476.   630.   592.   671. -8392. 14164.   404.  -606. -1314.
    272.]]


## Predict


In [6]:
def preprocess_features(X: np.ndarray) -> np.ndarray:
    if X.ndim != 2 or X.shape[1] != len(FEATURE_COLS):
        raise ValueError("expected shape (n, %d), got %s" % (len(FEATURE_COLS), X.shape))
    return X.astype(np.float64, copy=False)


def predict_scaled(X: np.ndarray):
    t0 = time.perf_counter()
    fn = getattr(scaler, "feature_names_in_", None)
    if fn is None:
        X_in = X
    else:
        try:
            import pandas as pd

            X_in = pd.DataFrame(X, columns=list(fn))
        except Exception:
            X_in = X
    xs = scaler.transform(X_in)
    y = clf.predict(xs)
    return y, time.perf_counter() - t0


def decode_preds(ix: np.ndarray):
    return list(label_encoder.inverse_transform(ix))


_enc0, _infer0 = predict_scaled(preprocess_features(first_feat))
labs0 = decode_preds(_enc0)

print("Single prediction:", labs0)
print("Inference timing (scaled+predict): %.5fs" % _infer0)
append_progress({"single_preds": labs0, "infer_s": _infer0})

bite_labels = []
bite_infer_s = float("nan")
if CSV_ROWS:
    parts = []
    for sym in ["ب", "ي", "ت"]:
        hit = next((r for r in CSV_ROWS if r.get("label") == sym), None)
        if hit is not None:
            parts.append(row_to_matrix(hit))
    if parts:
        bite_X = np.vstack(parts)
        be, bite_infer_s = predict_scaled(preprocess_features(bite_X))
        bite_labels = decode_preds(be)
        print("\nTriple-row bite sequence preds:", bite_labels)
        print("Bite inference timing: %.5fs" % bite_infer_s)
append_progress({"bite_preds": bite_labels})


Single prediction: ['أ']
Inference timing (scaled+predict): 1.69125s



Triple-row bite sequence preds: ['ب', 'ي', 'ت']
Bite inference timing: 0.23199s


## Egyptian mapping + sentence helpers


In [7]:
EG_LETTER = {
    "أ": "أَلِف",
    "ا": "أَلِف",
    "ب": "بِه",
    "ت": "تِه",
    "ث": "سِه",
    "ج": "جِيم",
    "ح": "حَا",
    "خ": "خَا",
    "د": "دَال",
    "ذ": "ذَال",
    "ر": "رَا",
    "ز": "زَاي",
    "س": "سِين",
    "ش": "شِين",
    "ص": "صَاد",
    "ض": "ضَاد",
    "ط": "طَا",
    "ظ": "ظَا",
    "ع": "عِين",
    "غ": "غِين",
    "ف": "فِه",
    "ق": "أَاف",
    "ك": "كَاف",
    "ل": "لَام",
    "م": "مِيم",
    "ن": "نُون",
    "ه": "هَا",
    "و": "وَاو",
    "ي": "يَا",
}

DIGIT_MAP = {
    "0": "صِفْر",
    "1": "وَاحِد",
    "2": "اِتْنِين",
    "3": "تَلَاتَه",
    "4": "أَرْبَعَه",
    "5": "خَمْسَه",
    "6": "سِتَّه",
    "7": "سَبْعَه",
    "8": "تَمَانْيَه",
    "9": "تِسْعَه",
}

AR_WORD_NUM = {
    "صفر": DIGIT_MAP["0"],
    "واحد": DIGIT_MAP["1"],
    "اثنين": DIGIT_MAP["2"],
    "ثلاثة": DIGIT_MAP["3"],
    "أربعة": DIGIT_MAP["4"],
    "خمسة": DIGIT_MAP["5"],
    "ستة": DIGIT_MAP["6"],
    "سبعة": DIGIT_MAP["7"],
    "ثمانية": DIGIT_MAP["8"],
    "تسعة": DIGIT_MAP["9"],
    "عشرة": "عَشَرَة",
}

# Optional Egyptian-friendly pronunciations for joined surface words (after concatenating predicted tokens).
EG_WORD_TTS = {
    "بيت": "بَيْت",
    "باب": "بَاب",
    "ماما": "مَامَا",
    # Overlap with spoken numerals / labels
    "خمسة": AR_WORD_NUM["خمسة"],
    "اثنين": AR_WORD_NUM["اثنين"],
    "ثلاثة": AR_WORD_NUM["ثلاثة"],
    "واحد": AR_WORD_NUM["واحد"],
}

# Full joined sentences (only when you have a hand-tuned line).
EG_SENTENCE_TTS = {
    "انا بيت": "أَنا بَيْت",
    "أنا بيت": "أَنا بَيْت",
}


def normalize_pred_token(pred) -> str:
    return str(pred).strip() if pred is not None else ""


def map_prediction_to_egyptian_pronunciation(pred) -> str:
    tok = normalize_pred_token(pred)
    if tok in EG_LETTER:
        return EG_LETTER[tok]
    if tok in AR_WORD_NUM:
        return AR_WORD_NUM[tok]
    if tok in DIGIT_MAP:
        return DIGIT_MAP[tok]
    if tok in ("آ", "إ", "ٱ"):
        return EG_LETTER.get("أ", tok)
    return tok


def map_joined_word_to_tts(word: str) -> str:
    w = normalize_pred_token(word)
    if not w:
        return w
    if w in EG_WORD_TTS:
        return EG_WORD_TTS[w]
    if w in AR_WORD_NUM:
        return AR_WORD_NUM[w]
    return w


def lightly_tashkeel(s: str) -> str:
    # Stub hook — extend if you integrate a tashkeeler later.
    return s


def finalize_xtts_utterance(s: str) -> str:
    """Exactly one trailing full stop — helps XTTS pause/stop cleanly (no narrator wrappers)."""
    t = lightly_tashkeel((s or "").strip())
    if not t:
        return t
    if t.endswith("."):
        return t
    return t + "."


def speak_single_prediction(pred) -> str:
    """TTS line: mapped pronunciation + optional full stop."""
    return finalize_xtts_utterance(map_prediction_to_egyptian_pronunciation(pred))


def speak_predictions_as_letters(predictions) -> str:
    """TTS line: pronounce each label (Arabic comma), one trailing full stop."""
    toks = [normalize_pred_token(x) for x in predictions if normalize_pred_token(x)]
    if not toks:
        return ""
    parts = [map_prediction_to_egyptian_pronunciation(t) for t in toks]
    return finalize_xtts_utterance("، ".join(parts[:50]))


def speak_predictions_as_joined_word(predictions) -> str:
    """Join tokens into one written word, EG_WORD_TTS if present; one trailing full stop."""
    toks = [normalize_pred_token(x) for x in predictions if normalize_pred_token(x)]
    surf = "".join(toks[:50])
    if not surf:
        return ""
    inner = EG_WORD_TTS[surf] if surf in EG_WORD_TTS else surf
    return finalize_xtts_utterance(inner)


def speak_predictions_as_sentence(predictions) -> str:
    """Sentence from tokens (+ spaces); EG_SENTENCE/word maps; one trailing full stop."""
    pieces: list[str] = []
    for x in predictions:
        if x is None:
            continue
        if isinstance(x, str) and x.strip() == "" and len(x) > 0:
            pieces.append(" ")
            continue
        t = normalize_pred_token(x)
        if t == "":
            continue
        if t.isspace():
            pieces.append(" ")
        else:
            pieces.append(t)
    joined = "".join(pieces)
    joined = " ".join(joined.split())
    if not joined:
        return ""
    if joined in EG_SENTENCE_TTS:
        return finalize_xtts_utterance(EG_SENTENCE_TTS[joined])
    words = joined.split()
    mapped = [map_joined_word_to_tts(w) for w in words]
    return finalize_xtts_utterance(" ".join(mapped))


SHORT_WARMUP_TEXT = speak_single_prediction(bite_labels[0] if bite_labels else labs0[0])

print("\nPRON (short, TTS-only):", SHORT_WARMUP_TEXT)



PRON (short, TTS-only): بِه.


## Helpers: VRAM cleanup


In [8]:
def unload_cuda(verbose: bool = True):
    gc.collect()
    if torch is not None and CUDA_AVAILABLE:
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
    if verbose:
        print("[gc+cuda-empty_cache]")


def format_bytes(nb: int) -> str:
    if nb < 1024:
        return f"{nb} B"
    if nb < 1024**2:
        return f"{nb / 1024:.2f} KB"
    return f"{nb / 1024 / 1024:.2f} MB"


In [9]:
def synth_xtts(phrases: dict[str, str], device_try: str):
    """Returns (paths_by_slug dict, timings dict, api object or None). Caller must delete api."""
    from IPython.display import Audio as IPA, display

    out: dict[str, str | None] = {}
    timings: dict = {"voice_model": "Egyptian_XTTS_local", "load_s": float("nan"), "device_used": "", "items": {}, "errs": {}}
    api = None
    if not HAVE_TTS or torch is None:
        timings["errs"]["fatal"] = "pip install TTS torch torchaudio soundfile"
        print(timings["errs"]["fatal"])
        return out, timings, None

    from TTS.api import TTS

    order = []
    if device_try == "cuda" and CUDA_AVAILABLE:
        order = ["cuda", "cpu"]
    else:
        order = ["cpu"]

    speaker_wav = str(XTTS_DIR / "speaker_reference.wav")
    # XTTS `load_checkpoint` treats the 2nd positional arg as a *directory*; passing .../model.pth breaks (joins model.pth twice).
    model_dir = str(XTTS_DIR)
    cfg_json = str(XTTS_DIR / "config.json")

    for dv in order:
        t_load0 = time.perf_counter()
        try:
            api = TTS(model_path=model_dir, config_path=cfg_json, gpu=dv.startswith("cuda"))
            timings["load_s"] = time.perf_counter() - t_load0
            timings["device_used"] = dv
            break
        except RuntimeError as rte:
            print(f" XTTS load error on {dv}:", rte)
            timings["errs"][f"rte_{dv}"] = repr(rte)
            if "out of memory" in repr(rte).lower():
                unload_cuda(False)
                api = None
                continue
            api = None
            break
        except Exception:
            timings["errs"][f"exc_{dv}"] = traceback.format_exc()
            print(traceback.format_exc())
            api = None
            break

    if api is None:
        return out, timings, None

    for slug, txt in phrases.items():
        wave_path = XTTS_AUDIO_DIR / f"{slug}.wav"
        g0 = time.perf_counter()
        meta = {"path": str(wave_path)}
        ok = False
        try:
            with torch.inference_mode():
                t_i0 = time.perf_counter()
                utter = finalize_xtts_utterance(txt)
                api.tts_to_file(
                    text=utter[:500],
                    file_path=str(wave_path),
                    speaker_wav=speaker_wav,
                    language="ar",
                )
                meta["inference_s"] = time.perf_counter() - t_i0
            meta["gen_s"] = time.perf_counter() - g0
            meta["bytes"] = wave_path.stat().st_size if wave_path.is_file() else 0
            out[slug] = str(wave_path.resolve())
            ok = True
            print(f" XTTS {slug}: {format_bytes(meta['bytes'])} | inference {meta.get('inference_s', float('nan')):.3f}s total {meta['gen_s']:.3f}s device {timings['device_used']}")
        except RuntimeError as rte:
            meta["error"] = repr(rte)
            timings["errs"][slug] = meta["error"][:240]
            if "out of memory" in meta["error"].lower():
                print(" XTTS CUDA OOM — fallback path: rerun cell forcing CPU manually if needed")
            out[slug] = None
        except Exception:
            meta["trace"] = traceback.format_exc()
            out[slug] = None
            print(meta["trace"][:600])
        timings["items"][slug] = meta

    append_progress({"xtts_timing": timings})
    # playback
    for slug, path in list(out.items()):
        if path:
            print(" Playback XTTS:", slug)
            display(IPA(filename=path))

    return out, timings, api


## Real Gloves Input Tests

Examples below are **copied from `CombinedDatasetModel.ipynb`** glove demo cells (same `GlovesInput` arrays and expected labels). They run through **this** notebook's scaler + model + Egyptian mapping + XTTS.

**TTS rule:** whatever is synthesized must be **only** the Egyptian-friendly mapped pronunciation (no “نُنَطِّق…”, “الكلمة…”, or other wrapper sentences).


In [10]:
# Source: CombinedDatasetModel.ipynb glove demo cells (read-only reference; not executed here)
REAL_GLOVES_CASES = [
    {
        "name": "GlovesInput_demo_أ",
        "expected": "أ",
        "values": [
            865, 472, 632, 593, 675,
            -8380, 14150, 410,
            -610, -1310, 275,
        ],
    },
    {
        "name": "GlovesInput_demo_اثنين",
        "expected": "اثنين",
        "values": [
            780, 400, 645, 782, 946,
            -8400, 14164, -1900,
            -100, -1500, -400,
        ],
    },
    {
        "name": "GlovesInput_demo_خمسة",
        "expected": "خمسة",
        "values": [
            910, 885, 945, 805, 960,
            1700, 16450, 450,
            100, -150, -400,
        ],
    },
]

REAL_GLOVE_RESULTS = []
REAL_GLOVE_XTTS_PHRASES = {}


def predict_from_glove_row(values: list) -> tuple:
    row = np.array([values], dtype=np.float64).reshape(1, -1)
    enc, t_infer = predict_scaled(preprocess_features(row))
    labs = decode_preds(enc)
    return enc[0], labs[0], t_infer


for i, case in enumerate(REAL_GLOVES_CASES, start=1):
    print("\n" + "-" * 40)
    print(f"Test #{i}")
    print("-" * 40)
    vals = case["values"]
    print("Input numbers:")
    print(vals)
    exp = case.get("expected")
    if exp is not None:
        print("\nExpected label:", exp)
    enc_i, pred, t_inf = predict_from_glove_row(vals)
    print("Raw encoded class index:", int(enc_i))
    print("Predicted label:", pred)
    mapped = map_prediction_to_egyptian_pronunciation(pred)
    print("Mapped pronunciation (Egyptian-friendly):", mapped)
    voice_txt = speak_single_prediction(pred)
    print("Final text sent to TTS:", voice_txt)
    slug = f"real_glove_{i}"
    REAL_GLOVE_XTTS_PHRASES[slug] = voice_txt[:500]
    REAL_GLOVE_RESULTS.append(
        {
            "test": i,
            "expected": exp,
            "predicted": pred,
            "mapped": mapped,
            "tts_text": voice_txt,
            "slug": slug,
            "infer_s": t_inf,
        }
    )
append_progress({"real_glove_results": [{"test": r["test"], "pred": r["predicted"]} for r in REAL_GLOVE_RESULTS]})
print("\nBuilt REAL_GLOVE_XTTS_PHRASES keys:", list(REAL_GLOVE_XTTS_PHRASES.keys()))
# XTTS + playback happens in the next cell together with supplementary phrases (single model load).



----------------------------------------
Test #1
----------------------------------------
Input numbers:
[865, 472, 632, 593, 675, -8380, 14150, 410, -610, -1310, 275]

Expected label: أ
Raw encoded class index: 0
Predicted label: أ
Mapped pronunciation (Egyptian-friendly): أَلِف
Final text sent to TTS: أَلِف.

----------------------------------------
Test #2
----------------------------------------
Input numbers:
[780, 400, 645, 782, 946, -8400, 14164, -1900, -100, -1500, -400]

Expected label: اثنين


Raw encoded class index: 2
Predicted label: اثنين
Mapped pronunciation (Egyptian-friendly): اِتْنِين
Final text sent to TTS: اِتْنِين.

----------------------------------------
Test #3
----------------------------------------
Input numbers:
[910, 885, 945, 805, 960, 1700, 16450, 450, 100, -150, -400]

Expected label: خمسة
Raw encoded class index: 12
Predicted label: خمسة
Mapped pronunciation (Egyptian-friendly): خَمْسَه
Final text sent to TTS: خَمْسَه.

Built REAL_GLOVE_XTTS_PHRASES keys: ['real_glove_1', 'real_glove_2', 'real_glove_3']


## XTTS synthesis (unload before Chatterbox)


In [11]:
_rg = globals().get("REAL_GLOVE_XTTS_PHRASES") or {}
PHR_MULTI = bite_labels[:] if bite_labels else labs0[: min(3, len(labs0))]

# Gesture-only demos — explicit letter / num / joined-word / sentence constructions
LETTER_SEQ_DEMO = ["أ", "ب", "ت"]
NUMBER_SEQ_DEMO = ["اثنين", "ثلاثة", "خمسة"]
JOIN_WORD_DEMO = ["ب", "ي", "ت"]
SENTENCE_DEMO = ["أ", "ن", "ا", " ", "ب", "ي", "ت"]

_xt_demo_lines = [
    ("short_probe", SHORT_WARMUP_TEXT),
    ("single_pred_voice", speak_single_prediction(labs0[0])),
    ("letters_list", speak_predictions_as_letters(LETTER_SEQ_DEMO)),
    ("bite_predictions_letters", speak_predictions_as_letters(PHR_MULTI[:3])),
    ("numbers_list", speak_predictions_as_letters(NUMBER_SEQ_DEMO)),
    ("joined_word", speak_predictions_as_joined_word(JOIN_WORD_DEMO)),
    ("sentence_demo", speak_predictions_as_sentence(SENTENCE_DEMO)),
]

print("\n--- XTTS phrase slugs → exact text ---")
for slug, t in _xt_demo_lines:
    print(f"  {slug}: {t}")

phrases_xtts = dict(_rg)
phrases_xtts.update(dict(_xt_demo_lines))

XTTS_PATHS, XTTS_TIMINGS, XTTS_API = synth_xtts(phrases_xtts, DEVICE_PREF)
globals()["GESTURE_TTS_DEMOS_DONE"] = True

for r in REAL_GLOVE_RESULTS:
    slug = r["slug"]
    p = XTTS_AUDIO_DIR / f"{slug}.wav"
    print("\n", "-" * 36)
    print(f"Test #{r['test']} — XTTS file:", p.resolve() if p.is_file() else "(missing)")

globals()["VERIFICATION_PLAYBACK_DONE"] = True



--- XTTS phrase slugs → exact text ---
  short_probe: بِه.
  single_pred_voice: أَلِف.
  letters_list: أَلِف، بِه، تِه.
  bite_predictions_letters: بِه، يَا، تِه.
  numbers_list: اِتْنِين، تَلَاتَه، خَمْسَه.
  joined_word: بَيْت.
  sentence_demo: أَنا بَيْت.


C:\Users\sondo\AppData\Local\Programs\Python\Python311\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


 > Using model: xtts


 > Text splitted to sentences.
['أَلِف.']


 > Processing time: 2.9183976650238037
 > Real-time factor: 2.223281803267512
 XTTS real_glove_1: 56.57 KB | inference 2.920s total 2.920s device cpu
 > Text splitted to sentences.
['اِتْنِين.']


...

 Playback XTTS: bite_predictions_letters


 Playback XTTS: numbers_list


 Playback XTTS: joined_word


 Playback XTTS: sentence_demo



 ------------------------------------
Test #1 — XTTS file: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts\real_glove_1.wav

 ------------------------------------
Test #2 — XTTS file: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts\real_glove_2.wav

 ------------------------------------
Test #3 — XTTS file: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts\real_glove_3.wav


In [12]:
try:
    del XTTS_API
except NameError:
    pass
finally:
    XTTS_API = None

unload_cuda()
print("XTTS unloaded; VRAM/cache cleared.")


[gc+cuda-empty_cache]
XTTS unloaded; VRAM/cache cleared.


## Chatterbox (Laghtna) — best-effort local load


In [13]:
CHATTER_PATHS: dict[str, str | None] = {}
CHATTER_TIMINGS: dict = {"voice_model": "laghtna_chatterbox_v1", "errs": {}, "items": {}}
# WAV slugs synthesized successfully this session (do not infer from leftover files alone)
CHATTERBOX_SUCCEEDED_SLUGS: set[str] = set()


def load_and_synth_chatterbox(phrases: dict[str, str]):
    CHATTER_PATHS.clear()
    CHATTER_TIMINGS.clear()
    CHATTER_TIMINGS["voice_model"] = "laghtna_chatterbox_v1_local"
    CHATTER_TIMINGS["errs"] = {}
    CHATTER_TIMINGS["items"] = {}

    from IPython.display import Audio as IPA, display

    if not (HAVE_CHATTERBOX and HAVE_SAFE and HAVE_HF and torch is not None):
        missing = []
        if not HAVE_CHATTERBOX:
            missing.append("pip install chatterbox-tts")
        if not HAVE_SAFE:
            missing.append("pip install safetensors")
        if not HAVE_HF:
            missing.append("pip install huggingface_hub")
        if torch is None:
            missing.append("pip install torch")
        CHATTER_TIMINGS["errs"]["deps"] = " | ".join(missing)
        print("Chatterbox skip:", CHATTER_TIMINGS["errs"]["deps"])
        return None

    try:
        import sys as _sy_mod

        try:
            import perth.perth_net as _pnp  # require `resemble-perth` (`pip uninstall -y perth` if conflict)
            _pm_root = _sy_mod.modules.get("perth")
            if _pm_root is not None and not callable(getattr(_pm_root, "PerthImplicitWatermarker", None)):
                setattr(_pm_root, "PerthImplicitWatermarker", _pnp.PerthImplicitWatermarker)
        except Exception as _pw:
            traceback.print_exc()
            print("Chatterbox perth watermark bridge failed:", _pw)
            print("Often fix: pip install resemble-perth  AND  pip install 'setuptools<81' (needs pkg_resources).")
            return None

        from huggingface_hub import hf_hub_download
        from safetensors.torch import load_file
        from chatterbox.mtl_tts import ChatterboxMultilingualTTS, Conditionals
        from chatterbox.models.t3 import T3
        from chatterbox.models.t3.modules.t3_config import T3Config
        from chatterbox.models.s3gen import S3Gen, S3GEN_SR
        from chatterbox.models.voice_encoder import VoiceEncoder
        from chatterbox.models.tokenizers.tokenizer import MTLTokenizer
    except ImportError:
        traceback.print_exc()
        return None

    try:
        grapheme_path = hf_hub_download(
            repo_id="ResembleAI/chatterbox",
            filename="grapheme_mtl_merged_expanded_v1.json",
            cache_dir=str(CACHE_DIR),
        )
    except Exception:
        CHATTER_TIMINGS["errs"]["hf_tokenizer"] = traceback.format_exc()
        print(CHATTER_TIMINGS["errs"]["hf_tokenizer"][-800:])
        return None

    order = []
    if DEVICE_PREF == "cuda" and CUDA_AVAILABLE:
        order = ["cuda", "cpu"]
    else:
        order = ["cpu"]

    api_obj = None
    load_wall = float("nan")
    for dv in order:
        t0 = time.perf_counter()
        try:
            map_loc = torch.device("cpu") if dv in ("cpu", "mps") else None
            ck = CHATTERBOX_DIR

            ve = VoiceEncoder()
            ve.load_state_dict(load_file(str(ck / "ve.safetensors")), strict=False)
            ve.eval()

            sg = S3Gen()
            res_sg = sg.load_state_dict(load_file(str(ck / "s3gen.safetensors")), strict=False)
            if getattr(res_sg, "missing_keys", ())[:1]:
                print("s3gen missing keys sample:", getattr(res_sg, "missing_keys", ())[:5])

            t3 = T3(T3Config.multilingual())
            t3.load_state_dict(load_file(str(ck / "t3_mtl23ls_v2.safetensors")), strict=False)

            ve = ve.to(dv).eval()
            sg = sg.to(dv).eval()
            t3 = t3.to(dv).eval()

            tokzr = MTLTokenizer(grapheme_path)
            cond_path = ck / "conds.pt"
            if cond_path.is_file():
                conds = Conditionals.load(str(cond_path), map_location=torch.device("cpu")).to(dv)
            else:
                conds = None

            api_obj = ChatterboxMultilingualTTS(t3, sg, ve, tokzr, dv, conds=conds)

            load_wall = time.perf_counter() - t0
            CHATTER_TIMINGS["load_s"] = load_wall
            CHATTER_TIMINGS["device_used"] = dv
            break

        except torch.cuda.OutOfMemoryError:
            CHATTER_TIMINGS["errs"][f"oom_load_{dv}"] = traceback.format_exc()
            unload_cuda(False)
            api_obj = None
            continue
        except Exception:
            CHATTER_TIMINGS["errs"][f"exc_load_{dv}"] = traceback.format_exc()
            print(traceback.format_exc())
            unload_cuda(False)
            api_obj = None
            continue

    if api_obj is None:
        print("Could not instantiate Chatterbox — see CHATTER_TIMINGS['errs'].")
        return None

    import inspect as _inspect

    try:
        from chatterbox.mtl_tts import SUPPORTED_LANGUAGES as _CB_LANGS
    except Exception:
        _CB_LANGS = type(api_obj).get_supported_languages()

    _gen_sig = _inspect.signature(api_obj.generate)

    dialect_aliases = {"ar-eg", "arq", "ar_eg", "eg", "egy"}
    langs_lower = {str(k).lower() for k in _CB_LANGS}
    dialect_hit = sorted(dialect_aliases & langs_lower)
    if dialect_hit:
        print("Chatterbox selected dialect/voice:", dialect_hit[0])
    else:
        print("No explicit Egyptian dialect setting found; using model default.")

    arabic_id = None
    for cand in ("ar", "arb"):
        if cand in _CB_LANGS:
            arabic_id = cand
            break
    if arabic_id is None:
        arabic_id = "ar"
    arabic_desc = _CB_LANGS.get(arabic_id, "") if arabic_id in _CB_LANGS else ""
    lbl = arabic_desc or "Arabic (multilingual pack)"
    print("Chatterbox language_id used:", f"{arabic_id} ({lbl})")

    _gen_kw = dict(
        language_id=arabic_id,
        audio_prompt_path=None,
        exaggeration=0.5,
        cfg_weight=0.45,
        temperature=0.75,
    )
    if "repetition_penalty" in _gen_sig.parameters:
        _rp = 1.2
        _gen_kw["repetition_penalty"] = _rp
        print("Chatterbox repetition_penalty used:", _rp, "(try 1.3–1.5 locally if chatter persists)")
    else:
        print("Chatterbox repetition_penalty used: unsupported")

    for slug, text in phrases.items():
        outp = CHATTER_AUDIO_DIR / f"{slug}.wav"
        ig0 = time.perf_counter()
        meta = {"path": str(outp.resolve()), "samplerate_hint": getattr(api_obj, "sr", None) or int(S3GEN_SR)}
        try:
            with torch.inference_mode():
                t_inf0 = time.perf_counter()
                wav_tensor = api_obj.generate(text[:500], **_gen_kw)
                meta["inference_s"] = time.perf_counter() - t_inf0
            sr = getattr(api_obj, "sr", None)
            if sr is None:
                sr = int(S3GEN_SR)

            wav = wav_tensor.squeeze(0).detach().cpu()
            wav_np = wav.numpy().astype("float32")

            wrote = False
            if HAVE_SF:
                try:
                    import soundfile as sf
                    sf.write(str(outp), wav_np, sr, subtype="PCM_16")
                    wrote = outp.is_file()
                except Exception:
                    wrote = False
            if not wrote:
                try:
                    import torchaudio as ta
                    ta.save(str(outp), wav.unsqueeze(0) if wav.dim() == 1 else wav, sr)
                    wrote = outp.is_file()
                except Exception:
                    traceback.print_exc()
            meta["gen_s"] = time.perf_counter() - ig0
            meta["bytes"] = outp.stat().st_size if outp.is_file() else 0
            CHATTER_PATHS[slug] = str(outp.resolve()) if wrote else None
            print(
                " Chatterbox",
                slug,
                "|",
                format_bytes(meta["bytes"]),
                "| infer:",
                meta.get("inference_s", float("nan")),
                "| device:",
                CHATTER_TIMINGS["device_used"],
            )
            CHATTER_TIMINGS["items"][slug] = meta
        except torch.cuda.OutOfMemoryError:
            CHATTER_TIMINGS["errs"][slug] = "OOM"
            unload_cuda(False)
            CHATTER_PATHS[slug] = None
        except Exception:
            CHATTER_TIMINGS["errs"][slug] = traceback.format_exc()
            CHATTER_TIMINGS["items"][slug] = {**meta, "fail": traceback.format_exc()[-900:]}
            traceback.print_exc()
            CHATTER_PATHS[slug] = None

    append_progress({"chatterbox_timing": dict(CHATTER_TIMINGS)})
    CHATTERBOX_SUCCEEDED_SLUGS |= {s for s, p in CHATTER_PATHS.items() if p}

    for slug, pth in CHATTER_PATHS.items():
        if pth:
            display(IPA(filename=pth))

    return api_obj


CHATTERBOX_API = load_and_synth_chatterbox(phrases_xtts.copy())


Chatterbox skip: pip install chatterbox-tts


In [14]:
try:
    del CHATTERBOX_API
except NameError:
    pass

unload_cuda()
print("Chatterbox unloaded.")


[gc+cuda-empty_cache]
Chatterbox unloaded.


## Comparison (same slug / same Egyptian text): XTTS vs Chatterbox


In [15]:
from IPython.display import Audio as IPA, display

comparison_rows = []

def info_line(name, slug, timings: dict):
    meta = timings.get("items", {}).get(slug, {})
    p = meta.get("path")
    b = meta.get("bytes", 0)
    infer = meta.get("inference_s", float("nan"))
    dev = timings.get("device_used", timings.get("device_used_final", ""))
    comparison_rows.append((name, slug, infer, dev, b, p))


def summarize_compare(slug):
    xt_line = XTTS_TIMINGS["items"].get(slug)
    cb_line = CHATTER_TIMINGS.get("items", {}).get(slug)
    print("\n===== COMPARE slug:", slug, "====")
    print(" Voice text used:", repr(phrases_xtts.get(slug, "")[:200]))
    xp = XTTS_PATHS.get(slug)
    cp = CHATTER_PATHS.get(slug)
    if xp:
        print(" XTTS path:", xp, "| bytes:", Path(xp).stat().st_size if Path(xp).is_file() else 0)
    else:
        print(" XTTS: (missing)")
    if cp:
        print(" Chatterbox path:", cp, "| bytes:", Path(cp).stat().st_size if Path(cp).is_file() else 0)
    else:
        print(" Chatterbox: (missing)")
    xt_inf = XTTS_TIMINGS.get("items", {}).get(slug, {}).get("inference_s", float("nan"))
    cb_inf = CHATTER_TIMINGS.get("items", {}).get(slug, {}).get("inference_s", float("nan"))
    print(f" Inference seconds — XTTS: {xt_inf:.4f}, Chatterbox: {cb_inf:.4f}")
    print(" Devices:", "XTTS", XTTS_TIMINGS.get("device_used"), "| CB", CHATTER_TIMINGS.get("device_used"))

    print("\n Playback — XTTS then Chatterbox")
    try:
        if xp and Path(xp).is_file():
            display(IPA(filename=xp))
        if cp and Path(cp).is_file():
            display(IPA(filename=cp))
    except Exception:
        pass


for slug in (
    "short_probe",
    "single_pred_voice",
    "letters_list",
    "bite_predictions_letters",
    "numbers_list",
    "joined_word",
    "sentence_demo",
):
    if slug in phrases_xtts:
        summarize_compare(slug)



===== COMPARE slug: short_probe ====
 Voice text used: 'بِه.'
 XTTS path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts\short_probe.wav | bytes: 62028
 Chatterbox: (missing)
 Inference seconds — XTTS: 2.4137, Chatterbox: nan
 Devices: XTTS cpu | CB None

 Playback — XTTS then Chatterbox



===== COMPARE slug: single_pred_voice ====
 Voice text used: 'أَلِف.'
 XTTS path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts\single_pred_voice.wav | bytes: 81996
 Chatterbox: (missing)
 Inference seconds — XTTS: 3.0159, Chatterbox: nan
 Devices: XTTS cpu | CB None

 Playback — XTTS then Chatterbox



===== COMPARE slug: letters_list ====
 Voice text used: 'أَلِف، بِه، تِه.'
 XTTS path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts\letters_list.wav | bytes: 430156
 Chatterbox: (missing)
 Inference seconds — XTTS: 16.6151, Chatterbox: nan
 Devices: XTTS cpu | CB None

 Playback — XTTS then Chatterbox



===== COMPARE slug: bite_predictions_letters ====
 Voice text used: 'بِه، يَا، تِه.'
 XTTS path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts\bite_predictions_letters.wav | bytes: 153676
 Chatterbox: (missing)
 Inference seconds — XTTS: 5.8743, Chatterbox: nan
 Devices: XTTS cpu | CB None

 Playback — XTTS then Chatterbox



===== COMPARE slug: numbers_list ====
 Voice text used: 'اِتْنِين، تَلَاتَه، خَمْسَه.'
 XTTS path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts\numbers_list.wav | bytes: 178252
 Chatterbox: (missing)
 Inference seconds — XTTS: 6.6506, Chatterbox: nan
 Devices: XTTS cpu | CB None

 Playback — XTTS then Chatterbox



===== COMPARE slug: joined_word ====
 Voice text used: 'بَيْت.'
 XTTS path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts\joined_word.wav | bytes: 73292
 Chatterbox: (missing)
 Inference seconds — XTTS: 2.6320, Chatterbox: nan
 Devices: XTTS cpu | CB None

 Playback — XTTS then Chatterbox



===== COMPARE slug: sentence_demo ====
 Voice text used: 'أَنا بَيْت.'
 XTTS path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts\sentence_demo.wav | bytes: 137804
 Chatterbox: (missing)
 Inference seconds — XTTS: 5.2974, Chatterbox: nan
 Devices: XTTS cpu | CB None

 Playback — XTTS then Chatterbox


## Gesture demos (letters / digits / joins)

Letter, number-sequence, joined-word, and short-sentence XTTS clips are synthesized in **“XTTS synthesis”** (`letters_list`, `bite_predictions_letters`, `numbers_list`, `joined_word`, `sentence_demo`). Property-/place-holder tashkeel experiments were removed — this notebook stays on the glove gesture pipeline only.



In [16]:
# Property-style tashkeel sweeps removed (were not part of the gesture project).
# Re-run the **XTTS synthesis** cell above if you need fresh letter / number / word / sentence clips.
print(
    "Gesture TTS demos live in slugs:",
    "letters_list, bite_predictions_letters, numbers_list, joined_word, sentence_demo",
)


Gesture TTS demos live in slugs: letters_list, bite_predictions_letters, numbers_list, joined_word, sentence_demo


## FINAL end-to-end (sample → pronounce → XTTS → Chatterbox)


In [17]:
from IPython.display import Audio as IPA, display

VERIFICATION_PLAYBACK_DONE = False


def run_final_pipeline_once():
    fr = preprocess_features(first_feat.copy())
    yhat, tinfer = predict_scaled(fr)
    pred_labs = decode_preds(yhat)
    full_text = speak_single_prediction(pred_labs[0])
    print("\n===== FINAL PIPELINE (dataset first row) =====")
    print("Sample input (features):", first_feat.ravel().tolist())
    print("Raw encoded prediction index:", int(yhat[0]))
    print("Decoded label:", pred_labs[0])
    mapped = map_prediction_to_egyptian_pronunciation(pred_labs[0])
    print("Mapped pronunciation:", mapped)
    print("Final text sent to TTS:", full_text)
    append_progress({"final_pred_labels": pred_labs, "final_voice_text_sample": full_text[:400]})

    unload_cuda(False)
    fph = {"pipeline_final_xt": full_text[:450]}
    pxt, mtx, axi = synth_xtts(fph, DEVICE_PREF)
    print("XTTS audio path:", list((pxt or {}).values())[0] if pxt else None)
    global VERIFICATION_PLAYBACK_DONE
    VERIFICATION_PLAYBACK_DONE = True
    try:
        del axi
    except Exception:
        pass
    unload_cuda(False)


run_final_pipeline_once()

# --- Final verification (Chatterbox line only if playable audio was written for glove slugs) ---
from pathlib import Path

_ok_pred = clf is not None and scaler is not None and label_encoder is not None
_ok_multi = len(REAL_GLOVE_RESULTS) >= 3

_BAD_TTS_FRAGMENTS = (
    "نُنَطِّق",
    "الكلمة:",
    "النتيجة هي",
    "الرمز المتوقع",
    "المخرج غير متوفر",
)

_ok_tts_clean = "phrases_xtts" in globals() and all(
    not any(b in (txt or "") for b in _BAD_TTS_FRAGMENTS) for txt in phrases_xtts.values()
)


def _wav_ok(p: Path) -> bool:
    return p.is_file() and p.stat().st_size > 1000


def _xtts_real_ok() -> bool:
    return all(_wav_ok(XTTS_AUDIO_DIR / f"{r['slug']}.wav") for r in REAL_GLOVE_RESULTS)


_gesture_lws = {"letters_list", "joined_word", "sentence_demo"}

_ok_gesture_lws = globals().get("GESTURE_TTS_DEMOS_DONE") and all(
    _wav_ok(XTTS_AUDIO_DIR / f"{s}.wav") for s in _gesture_lws
)

_ok_xtts = _xtts_real_ok() and _ok_gesture_lws
_play = globals().get("VERIFICATION_PLAYBACK_DONE", False)

print("\n" + "=" * 52)
print("FINAL VERIFICATION")
print("=" * 52)
print("✅ Real glove tests executed" if _ok_multi else "❌ Real glove tests")
print("✅ TTS uses ONLY mapped pronunciation" if _ok_tts_clean else "❌ TTS wrapper text leaked (inspect phrases_xtts)")
print("✅ XTTS audio generated" if _ok_xtts else "❌ XTTS audio (check generated_audio/xtts/)")
print("✅ Audio playback displayed" if _play else "⚠ Audio widgets (visible in Jupyter; nbconvert may omit widgets)")
print(
    "✅ Letter/word/sentence tests executed"
    if globals().get("GESTURE_TTS_DEMOS_DONE")
    else "❌ Letter/word/sentence tests",
)

_need_cb = {r["slug"] for r in REAL_GLOVE_RESULTS}
_cb_ok = bool(
    _need_cb
    and (_need_cb <= globals().get("CHATTERBOX_SUCCEEDED_SLUGS", set()))
    and all(_wav_ok(CHATTER_AUDIO_DIR / f"{s}.wav") for s in _need_cb)
)
print("✅ Chatterbox working" if _cb_ok else "(Chatterbox skipped or failed — see errors above)")
print("=" * 52)



===== FINAL PIPELINE (dataset first row) =====
Sample input (features): [861.0, 476.0, 630.0, 592.0, 671.0, -8392.0, 14164.0, 404.0, -606.0, -1314.0, 272.0]
Raw encoded prediction index: 0
Decoded label: أ
Mapped pronunciation: أَلِف
Final text sent to TTS: أَلِف.


 > Using model: xtts


 > Text splitted to sentences.
['أَلِف.']


 > Processing time: 2.416322946548462
 > Real-time factor: 1.8407932895036478
 XTTS pipeline_final_xt: 56.57 KB | inference 2.414s total 2.414s device cpu
 Playback XTTS: pipeline_final_xt


XTTS audio path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts\pipeline_final_xt.wav



FINAL VERIFICATION
✅ Real glove tests executed
✅ TTS uses ONLY mapped pronunciation
✅ XTTS audio generated
✅ Audio playback displayed
✅ Letter/word/sentence tests executed
(Chatterbox skipped or failed — see errors above)
